In [1]:
import os
import csv

import torch
import torch.nn as nn

from torchvision.models import vit_b_32
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
def create_vit_model(num_encoder_layers=2, num_classes=2):
    model = vit_b_32(weights=None)

    model.encoder.layers = nn.Sequential(
        *[
            model.encoder.layers[i]
            for i in range(num_encoder_layers)
        ]
    )

    model.heads.head = nn.Linear(
        model.heads.head.in_features,
        num_classes
    )

    return model


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = create_vit_model(
    num_encoder_layers=2,
    num_classes=2
)

model = model.to(device)

print("Model architecture created")

Using device: cuda
GPU: NVIDIA A10G


Model architecture created


In [3]:
MODEL_PATH = "/home/sagemaker-user/best_vit_model_full_scratch.pth"

assert os.path.exists(MODEL_PATH), (
    f"Model file not found: {MODEL_PATH}"
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Loaded:", MODEL_PATH)

Loaded: /home/sagemaker-user/best_vit_model_full_scratch.pth


In [8]:
TEST_DIR = "/mnt/sagemaker-nvme/vit_full_dataset/Test"

print("Test directory exists:", os.path.exists(TEST_DIR))

if os.path.exists(TEST_DIR):
    print(
        "Test/down:",
        len(os.listdir(f"{TEST_DIR}/down"))
    )
    print(
        "Test/up:",
        len(os.listdir(f"{TEST_DIR}/up"))
    )

Test directory exists: True


Test/down: 13382203


Test/up: 15948847


In [6]:
import os

TEST_DIR = "/mnt/sagemaker-nvme/vit_full_dataset/Test"

print(os.path.exists(TEST_DIR))

True


In [9]:
import os

print("Down:", len(os.listdir("/mnt/sagemaker-nvme/vit_full_dataset/Test/down")))
print("Up:", len(os.listdir("/mnt/sagemaker-nvme/vit_full_dataset/Test/up")))

Down: 13382203


Up: 15948847


In [11]:
import os

TEST_DIR = "/mnt/sagemaker-nvme/vit_full_dataset/Test"

print("Classes:", os.listdir(TEST_DIR))
print("Down folder exists:", os.path.isdir(f"{TEST_DIR}/down"))
print("Up folder exists:", os.path.isdir(f"{TEST_DIR}/up"))

Classes: ['down', 'up']
Down folder exists: True
Up folder exists: True


In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Test images:", len(test_dataset))
print("Classes:", test_dataset.classes)
print("Test batches:", len(test_loader))

Test images: 29331050
Classes: ['down', 'up']
Test batches: 916596


In [13]:
import os

print("Down:", len(os.listdir(f"{TEST_DIR}/down")))
print("Up  :", len(os.listdir(f"{TEST_DIR}/up")))
print("Total:", len(os.listdir(f"{TEST_DIR}/down")) +
               len(os.listdir(f"{TEST_DIR}/up")))

Down: 13382203


Up  : 15948847


Total: 29331050


In [11]:
import os
import torch
import torch.nn as nn
from torchvision.models import vit_b_32

def create_vit_model(num_encoder_layers=2, num_classes=2):
    model = vit_b_32(weights=None)

    model.encoder.layers = nn.Sequential(
        *[
            model.encoder.layers[i]
            for i in range(num_encoder_layers)
        ]
    )

    model.heads.head = nn.Linear(
        model.heads.head.in_features,
        num_classes
    )

    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = create_vit_model(
    num_encoder_layers=2,
    num_classes=2
)

MODEL_PATH = "/home/sagemaker-user/best_vit_model_full_scratch.pth"

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

print("Model loaded successfully.")

Model loaded successfully.


In [13]:
import csv
import torch

@torch.no_grad()
def stream_test_to_csv(
    model,
    loader,
    criterion,
    device,
    output_csv
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    dataset = loader.dataset

    current_index = 0

    with open(output_csv, "w", newline="") as csvfile:

        writer = csv.writer(csvfile)

        writer.writerow([
            "path",
            "true_label",
            "predicted_label",
            "prob_down",
            "prob_up"
        ])

        for batch_idx, (images, labels) in enumerate(loader):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1)

            prob_down = probs[:,0].cpu().numpy()
            prob_up = probs[:,1].cpu().numpy()

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            preds = preds.cpu().numpy()
            labels_np = labels.cpu().numpy()

            batch_size = len(labels_np)

            paths = [
                dataset.samples[current_index + i][0]
                for i in range(batch_size)
            ]

            current_index += batch_size

            rows = zip(
                paths,
                labels_np,
                preds,
                prob_down,
                prob_up
            )

            writer.writerows(rows)

            if batch_idx % 1000 == 0:

                csvfile.flush()

                print(
                    f"Batch {batch_idx:,}/{len(loader):,} "
                    f"| Images: {total:,}"
                )

    test_loss = running_loss / total
    test_acc = correct / total

    print("\nCSV completed.")

    return test_loss, test_acc
    

In [14]:
OUTPUT_CSV = "/home/sagemaker-user/vit_signals_full_scratch.csv"

test_loss, test_acc = stream_test_to_csv(
    model,
    test_loader,
    criterion,
    device,
    OUTPUT_CSV
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"CSV saved to: {OUTPUT_CSV}")

Batch 0/916,596 | Images: 32


Batch 1,000/916,596 | Images: 32,032


Batch 2,000/916,596 | Images: 64,032


Batch 3,000/916,596 | Images: 96,032


Batch 4,000/916,596 | Images: 128,032


Batch 5,000/916,596 | Images: 160,032


... (20 outputs left out)

Batch 26,000/916,596 | Images: 832,032


Batch 27,000/916,596 | Images: 864,032


Exception in thread Thread-5 (_pin_memory_loop):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/opt/conda/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/mu

KeyboardInterrupt: 

In [15]:
from torch.utils.data import Subset, DataLoader

CSV_PATH = "/home/sagemaker-user/vit_signals_full_scratch.csv"

completed_images = 872160

remaining_dataset = Subset(
    test_dataset,
    range(completed_images, len(test_dataset))
)

resume_loader = DataLoader(
    remaining_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Already completed:", completed_images)
print("Remaining images:", len(remaining_dataset))
print("Remaining batches:", len(resume_loader))

Already completed: 872160
Remaining images: 28458890
Remaining batches: 889341


In [16]:
import csv
import torch

@torch.no_grad()
def resume_test_to_csv(
    model,
    loader,
    criterion,
    device,
    output_csv,
    start_index
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    base_dataset = loader.dataset.dataset

    with open(output_csv, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)

        current_index = start_index

        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probabilities = torch.softmax(outputs, dim=1)

            prob_down = probabilities[:, 0].cpu().numpy()
            prob_up = probabilities[:, 1].cpu().numpy()

            predicted_labels = torch.argmax(outputs, dim=1)

            correct += (predicted_labels == labels).sum().item()
            total += labels.size(0)

            predicted_labels = predicted_labels.cpu().numpy()
            true_labels = labels.cpu().numpy()

            batch_size = len(true_labels)

            paths = [
                base_dataset.samples[current_index + i][0]
                for i in range(batch_size)
            ]

            current_index += batch_size

            writer.writerows(
                zip(
                    paths,
                    true_labels,
                    predicted_labels,
                    prob_down,
                    prob_up
                )
            )

            if batch_idx % 1000 == 0:
                csvfile.flush()

                print(
                    f"Resume Batch {batch_idx:,}/{len(loader):,} | "
                    f"Total images in CSV: {current_index:,}"
                )

    test_loss = running_loss / total
    test_acc = correct / total

    return test_loss, test_acc

In [17]:
print("resume_test_to_csv" in globals())

True


In [18]:
resume_loss, resume_acc = resume_test_to_csv(
    model,
    resume_loader,
    criterion,
    device,
    CSV_PATH,
    completed_images
)

print(f"\nResume Loss: {resume_loss:.4f}")
print(f"Resume Accuracy: {resume_acc:.4f}")
print(f"CSV updated at: {CSV_PATH}")

Resume Batch 0/889,341 | Total images in CSV: 872,192


Resume Batch 1,000/889,341 | Total images in CSV: 904,192


Resume Batch 2,000/889,341 | Total images in CSV: 936,192


Resume Batch 3,000/889,341 | Total images in CSV: 968,192


Resume Batch 4,000/889,341 | Total images in CSV: 1,000,192


Resume Batch 5,000/889,341 | Total images in CSV: 1,032,192


... (881 outputs left out)

Resume Batch 887,000/889,341 | Total images in CSV: 29,256,192


Resume Batch 888,000/889,341 | Total images in CSV: 29,288,192


Resume Batch 889,000/889,341 | Total images in CSV: 29,320,192



Resume Loss: 0.6897
Resume Accuracy: 0.5366
CSV updated at: /home/sagemaker-user/vit_signals_full_scratch.csv


In [21]:
import pandas as pd

df = pd.read_csv("/home/sagemaker-user/vit_signals_full_scratch.csv")

print(len(df))